In [ ]:
from common import *

## 8. Detekcija i uklanjanje anomalija
### 8.1 Univarijantna analiza anomalija

U priloženom kodu primenjen je robusni neparametarski pristup za univarijatnu detekciju anomalija, potpuno usklađen sa izrazito asimetričnom prirodom meteoroloških podataka. Pošto je prethodna statistička analiza jasno pokazala odsustvo normalne raspodele, metodologija se oslanja na dve stabilne neparametarske metrike: 
1. Modifikovani Z‑skor zasnovan na MAD‑u(Median absolute diviation - medijana apsolutnih razlika određene vrednosti i medijane grupe), nakon izračunavanja MAD-a, za svaku tačku se računa z score po standardnoj formuli uz izmenu da se umesto srednje vrednosti koristi MAD. Tako dobijena vrednost množi se odgovarajućim koeficijentom kako bi se mogla porediti sa vrednostima normalne raspodela.
2. Kontinualni IQR‑skor - ovaj score nam govori koliko interkvartilnih raspona je određena vrednost udaljena od medijane - idealna vrednost je 0 a sve vrednosti van intervala (-1,1) se mogu smatrati anomalijom.  

Sve statističke vrednosti izračunavaju se lokalno, grupisanjem podataka prema geografskoj lokaciji i kalendarskom mesecu, čime se čuvaju sezonske i prostorne karakteristike klimatskih obrazaca.

U kod je dodat i mehanizam zaštite od deljenja nulom, koji se aktivira u slučajevima kada su vrednosti u posmatranoj grupi uniformne, što je česta pojava kod određenih meteoroloških atributa. Zadržavanje kontinualnih skorova (umesto binarne klasifikacije anomalija) nam omogućava daju analizu dobijenih rezultata nakon izvršavanja svih preostalih testova, rezultati će biti sačuvani u zaseban csv fajl.

In [ ]:
data = loadData("backups/weatherAUSAfter5_2.csv")
is_train = data['Date'] < SPLIT_DATE
identifiers = data[['Location', 'Date']].copy()
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['Month', 'Year', "Day"]]

final_results = identifiers.copy()

for target_col in numeric_cols:
    data2 = data[['Location', 'Month', target_col]]
    train_data = data2[is_train].dropna()

    if train_data.empty: continue

    grouped = train_data.groupby(['Location', 'Month'])[target_col]

    median_lookup = grouped.median()
    mad_lookup = grouped.apply(lambda x: np.median(np.abs(x - np.median(x))))
    q25_lookup = grouped.quantile(0.25)
    q75_lookup = grouped.quantile(0.75)

    global_median = train_data[target_col].median()
    global_mad = np.median(np.abs(train_data[target_col] - global_median))
    global_q25 = train_data[target_col].quantile(0.25)
    global_q75 = train_data[target_col].quantile(0.75)

    key = list(zip(data2['Location'], data2['Month']))
    median = pd.Series([median_lookup.get(k, global_median) for k in key], index=data2.index)
    mad = pd.Series([mad_lookup.get(k, global_mad) for k in key], index=data2.index)
    q25 = pd.Series([q25_lookup.get(k, global_q25) for k in key], index=data2.index)
    q75 = pd.Series([q75_lookup.get(k, global_q75) for k in key], index=data2.index)

    mad_safe = mad.replace(0, 1e-6)
    iqr_safe = (q75 - q25).replace(0, 1e-6)

    mod_z_score = 0.6745 * (data2[target_col] - median) / mad_safe
    iqr_score = (data2[target_col] - median) / iqr_safe

    valid = data2[target_col].notna()
    final_results.loc[valid, f'{target_col}_IQR_Score'] = iqr_score[valid]
    final_results.loc[valid, f'{target_col}_ModZ_Score'] = mod_z_score[valid]

final_results.to_csv('AnomalyDetectionResults/skorovi_univarijatni.csv', index=False)
print("Svi lokalizovani univarijatni skorovi su sačuvani u 'AnomalyDetectionResults/skorovi_univarijatni.csv'.")

### 8.2 Multivarijantna analiza anomalija

Ova analiza se bazira na ispitivanju odnosa između promenljivih. Njome ćemo pokušati da zaključimo postoje li vrednosti dva prediktora koje ne prate standardni obrazac raspodele. Na ovaj način utvrdićemo postoje li neobične kombinacije vrednosti prediktora za jedno merenje.

* **Mahalanobisova udaljenost** - Ovaj algoritam meri udaljenost tačke definisane sa $n$ atributa od opšteg srednjeg trenda (centra) raspodele. Za razliku od standardne euklidske razdaljine, Mahalanobisova udaljenost uzima u obzir korelaciju između svih atributa. Ako postoji jaka korelacija između dva atributa, odstupanje oba atributa od generalnog trenda se ne smatra toliko čudnim kao odstupanje u samo jednom atributu. Ovaj efekat usklađivanja dimenzija postiže se primenom inverzne matrice kovarijanse. S obzirom na to da ekstremne anomalije mogu imati nesrazmerno veliki uticaj na izračunavanje standardnog proseka i matrice kovarijanse, u ovoj analizi primenjena je robusna procena matrice kovarijanse (metod Minimum Covariance Determinant). Ovako formirana matrica teži da isprati centralnu distribuciju regularnih podataka, efikasno ignorišući uticaj autlajera prilikom definisanja referentnog centra, čime se značajno povećava preciznost i pouzdanost detekcije anomalija u višedimenzionalnom prostoru.
* **KNN** - iako se KNN inače koristi za predikciju neke vrednosti na osnovu k najbližih tačaka, mi ćemo ovaj algoritam iskoristiti kako bismo izračunali prosečnu udaljenost jedne tačke od njenih k (opredelili smo se za k=5) najbližih suseda. Što je ova distanca veća to je veća verovatnoća i da je neka tačka anomalija. Zbog potrebe za računanjem razdaljine između svih 140 hiljada tačaka, dodaćemo algorithm='auto' kako bi algoritam automatski zaključio kako je računanje tih distanci optimalno.
* **Isolation forest** - ovo je algoritam zasnovan na nenadgledanom masinskom učenju u predstavlja specifičnu varijantu algoritma Random forest za namene detekcija anomalija. Osnovna ideja algoritam je da kreira veći broj stabala odlučivanja i u svakom stablu pokušava da izoluje podatke u zasebne grane. oOni podaci koji su anomalije biće izolovani mnogo ranije od normalnih podataka. Izlaz ovog algoritma je realan broj (najčešće između -1 i 1 ali može i da varira). Što je broj niži to znači da je podatak bilo lakše izolovati i da je veća verovatnoća da je u pitanju anomalija. Zbog konzistentnosi sa rezultatima drugih modela koji vrćaju visoke vrednosti za anomalije, rezultat izolation forest-a ćemo pomnožiti sa (-1).

S obzirom na to da će većina ovih algoritama prijaviti grešku za nedostajuće vrednosti, privremeno ćemo sve takve pojave popuniti medijanom za datu lokaciju. Takođe izvšićemo skaliranje podataka (svođenje na N(0,1) raspodelu) kako bi algoritam dao bolje rezultate.

Kao i kod univarijantne analize podataka, rezulate ćemo smestiti u zaseban csv fajl radi dalje analize.


In [ ]:
existing_cols = ['MinTemp', 'MaxTemp', 'Rainfall', 'WindSpeed9am']

results = data[['Location', 'Date']].copy()

data_imputed_df = data.copy()


for col in existing_cols:
    loc_month_median = data_imputed_df.loc[is_train].groupby(['Location', 'Month'])[col].median()
    loc_median = data_imputed_df.loc[is_train].groupby('Location')[col].median()
    global_median = data_imputed_df.loc[is_train, col].median()

    key = list(zip(data_imputed_df['Location'], data_imputed_df['Month']))
    fill_lm = pd.Series([loc_month_median.get(k, np.nan) for k in key], index=data_imputed_df.index)
    fill_l = data_imputed_df['Location'].map(loc_median)
    data_imputed_df[col] = data_imputed_df[col].fillna(fill_lm).fillna(fill_l).fillna(global_median)

scaler = StandardScaler()
train_mat = scaler.fit_transform(data_imputed_df.loc[is_train, existing_cols])
data_mat = scaler.transform(data_imputed_df[existing_cols])

# --- Mahalanobisova udaljenost (fit na train, score na svima) ---
robust_cov = MinCovDet(support_fraction=0.75, random_state=42).fit(train_mat)
mahalanobis_distances = np.sqrt(robust_cov.mahalanobis(data_mat))
results['Mahalanobis_Score'] = mahalanobis_distances


k = 5
nbrs = NearestNeighbors(n_neighbors=k + 1, algorithm='auto').fit(train_mat)
distances, _ = nbrs.kneighbors(data_mat)
self_match = np.isclose(distances[:, 0], 0.0)
knn_dist_bez_sebe = np.where(self_match[:, None], distances[:, 1:], distances[:, :-1])
results['KNN_Score'] = knn_dist_bez_sebe.mean(axis=1)

iso_forest = IsolationForest(random_state=42)
iso_forest.fit(train_mat)

results['IForest_Score'] = -iso_forest.decision_function(data_mat)

results.to_csv('AnomalyDetectionResults/skorovi_multi.csv', index=False)
print("Multivarijatni skorovi su sačuvani u 'AnomalyDetectionResults/skorovi_multi.csv'.")

Pored navedenih metoda, upotrebićemo i LOF (Local Outlier Factor). Ovo je algoritam sličan KNN algoritmu uz bitnu promenu - ovaj algoritam poredi lokalnu gustinu posmatrane tačke sa lokalnom gustinom njenih suseda. Ukoliko je gustina tačke znatno manja od gustine njenih suseda, tačka se klasifikuje kao lokalni izolovani podatak (outlier).
Prednost ovog algoritma je sto je u mogucnosti da detektuje grupu tacaka koje su izolovane od opšteg trenda podataka ali zajedno grupišu klaster koji potencijalno može biti i novitet.
Kako ovaj algoritam računa gustine svih suseda, on je znatno memorijski i vremenski zahtevniji od već računski kompleksnog KNN algoritma. Zbog toga, izršićemo uzorkovanje na 40.000 elemenata na kojima ćemo trenirati naš model.

In [ ]:
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
df_numeric = data[['Location', 'Date'] + numeric_cols].copy()

df_valid = df_numeric.dropna().copy()
is_train_valid = is_train.reindex(df_valid.index)

scaler = StandardScaler()
train_scaled = scaler.fit_transform(df_valid.loc[is_train_valid, numeric_cols])
scaled_data = scaler.transform(df_valid[numeric_cols])

print("Treniranje LOF modela na uzorku (samo iz train dela)...")

np.random.seed(42)
sample_indices = np.random.choice(len(train_scaled), min(40000, len(train_scaled)), replace=False)
sample_data = train_scaled[sample_indices]

lof = LocalOutlierFactor(n_neighbors=35, novelty=True)
lof.fit(sample_data)

lof_scores = -lof.score_samples(scaled_data)

results = df_valid[['Location', 'Date']].copy()

results['LOF_Score'] = lof_scores

results.to_csv('AnomalyDetectionResults/skorovi_lof.csv', index=False)
print("Kontinualni LOF skorovi su izvezeni u 'AnomalyDetectionResults/skorovi_lof.csv'.")

### 8.3 Finalna detekcija i brisanje anomalije

Sada kada smo izvršili sve analize koje smo hteli učitaćemo sva tri izveštaja i spojiti ih u jedan dataframe, tako da u jednoj vrsti imamo sve informacije za dati datum i lokaciju.
Prethodno primenjene testove možemo podeliti na tri kategorije na osnovu njihove pouzdanosti:
1. Robusni statistički modeli otporni na uticaj anomalija koje možemo smatrati najpouzdanijim
2. Z score modeli koji iako su statistiki egzaktni mogu da budu veoma pristrasni prema anomalijama
3. Modeli mašinskog učenja koji su skloni lažnim signalima po pitanju anomalija

Zbog toga, ovim grupama testova dodelićemo različite vrednosti pondera da bismo naglasili njihovu pouzdanost. 
Za svaku vrstu u dataframe-u računami zbir svih skorova pomnoženih odgovarajućim ponderom, zatim vršimo dve transformacije:
* Računanje magnitude za svaku vrednost (percentila vrednosti ukupnog rezultata) - ova tranformacija će nam dati broj u opsegu od 0 do 1, što je veća vrednost - veća je verovatnoća da je podatak anomalija.
* Logaritamska transformacija: logaritam ima takvu osobinu da će veće vrednosti pojačati - ovim naglašavamo anomalije

Na kraju, uzimamo jedan procenat najekstremnijih vrednosti i brišemo ih uz pretpostavku da su u pitanju anomalije.

Pošto ovim završavamo deo vazan za detekciju anomalija, novodobijeni dataset ćemo izvesti u csv kako bi dalji rad učinili jednostavnijim.

In [ ]:
df_uni = pd.read_csv('AnomalyDetectionResults/skorovi_univarijatni.csv')
df_lof = pd.read_csv('AnomalyDetectionResults/skorovi_lof.csv')
df_multi = pd.read_csv('AnomalyDetectionResults/skorovi_multi.csv')

for d in [df_uni, df_lof, df_multi]:
    d['Date'] = pd.to_datetime(d['Date'], errors='coerce')

merged = df_uni.merge(df_multi, on=['Location', 'Date'], how='left')
merged = merged.merge(df_lof, on=['Location', 'Date'], how='left')

score_cols = [c for c in merged.columns if c not in ['Location', 'Date']]

def dobij_tezinu(ime_kolone):
    if 'ModZ' in ime_kolone or 'Mahalanobis' in ime_kolone or 'IQR' in ime_kolone:
        return 1.5
    if 'Z_Score' in ime_kolone:
        return 1.2
    if 'IForest' in ime_kolone or 'LOF' in ime_kolone or 'KNN' in ime_kolone:
        return 1.0
    return 1.0


final_scores = pd.Series(0.0, index=merged.index)
total_weights = pd.Series(0.0, index=merged.index)

for col in score_cols:
    w = dobij_tezinu(col)
    rank = merged[col].rank(pct=True)
    boosted = -np.log1p(-rank.clip(upper=0.9999))
    valid_mask = merged[col].notna()
    final_scores[valid_mask] += boosted[valid_mask] * w
    total_weights[valid_mask] += w

merged['Anomalijski_Skor'] = final_scores / total_weights.replace(0, np.nan)

merged_is_train = merged['Date'] < SPLIT_DATE
prag = merged.loc[merged_is_train, 'Anomalijski_Skor'].quantile(0.999)
merged['JE_ANOMALIJA'] = merged['Anomalijski_Skor'] > prag
obrisi = merged['JE_ANOMALIJA'] & merged_is_train

final_report = merged[merged['JE_ANOMALIJA']].sort_values(by='Anomalijski_Skor', ascending=False)
final_report.to_csv('AnomalyDetectionResults/konacni_izveštaj_anomalija.csv', index=False)
broj_anomalija = len(final_report)
broj_obrisanih = int(obrisi.sum())
print(f"\nGotovo! Prag detekcije je postavljen na: {prag:.4f} (racunat samo na train delu)")
print(f"Pronađeno je ukupno {broj_anomalija} anomalija (u celom skupu), od cega ce biti obrisano {broj_obrisanih} (samo iz train dela).")
print("Detaljan spisak pravih anomalija izvezen je u 'AnomalyDetectionResults/konacni_izveštaj_anomalija.csv'.")

mergedInfo = merged[['Location', 'Date']].copy()
mergedInfo['OBRISI'] = obrisi.values
cd_join = data.merge(mergedInfo, on=['Location', 'Date'], how='left')

cleaned_without_anoms = cd_join[cd_join['OBRISI'] != True].copy()

cleaned_without_anoms.drop(columns=['OBRISI'], inplace=True)

write_log(cleaned_without_anoms, "Obrisane anomalije (samo iz train dela)", "weatherAusAfter8_3.csv")